## What this notebook does

This notebook loads the SWORD river reaches layer and the Dnipro basin boundary, checks and aligns their coordinate reference systems, dissolves the basin into a single geometry, and clips the reaches to the basin extent. It then adds a `url` field for each clipped reach using its `reach_id`, saves the clipped reaches as a Shapefile, and exports a WGS84 GeoJSON version for use in web maps.

This GitHub-ready copy uses project-relative file paths so it can be shared without machine-specific local directories.


In [1]:
import geopandas as gpd
from pathlib import Path

# ----------------------------
# Paths
# ----------------------------
reaches_path = Path("data/shp/EU/eu_sword_reaches_hb22_v17b.shp")
basin_path   = Path("data/Dnipro_basin_shapefiles/mrb_basins.json")

out_dir = Path("data/Dnipro_basin_shapefiles/derived")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "dnipro_sword_reaches_clip.csv"

out_shp = out_dir / "dnipro_sword_reaches_clip.shp"     # Shapefile output
# out_gpkg = out_dir / "dnipro_sword_reaches_clip.gpkg" # Alternative (recommended for robustness)

# ----------------------------
# Read data
# ----------------------------
reaches = gpd.read_file(reaches_path)
basin = gpd.read_file(basin_path)

print("Reaches CRS:", reaches.crs)
print("Basin CRS:", basin.crs)

# ----------------------------
# Ensure basin has CRS; your earlier code suggests it should be EPSG:4326
# ----------------------------
if basin.crs is None:
    basin = basin.set_crs(epsg=4326)

# ----------------------------
# Make basin a single geometry (dissolve)
# ----------------------------
basin_union = basin.dissolve()  # 1-row GeoDataFrame
# (Optional) fix invalid geometries if needed
basin_union["geometry"] = basin_union.geometry.buffer(0)

# ----------------------------
# Reproject basin to reaches CRS (clip requires same CRS)
# ----------------------------
if reaches.crs is None:
    raise ValueError("Reaches layer has no CRS. Cannot safely clip without CRS.")

basin_union = basin_union.to_crs(reaches.crs)

# ----------------------------
# Clip
# ----------------------------
reaches_clip = gpd.clip(reaches, basin_union)

print("Input reaches:", len(reaches))
print("Clipped reaches:", len(reaches_clip))

# ----------------------------
# Add URL column
# ----------------------------
base_url = "https://ihpwinsdata.blob.core.windows.net/data/SWOT/SWOT_dnipro_reach_hydrocron_DAWG/reach_"

# Ensure reach_id is string (important if it's numeric)
reaches_clip["reach_id"] = reaches_clip["reach_id"].astype(str)

# Create url column
reaches_clip["url"] = base_url + reaches_clip["reach_id"] + ".csv"

# ----------------------------
# Save
# ----------------------------
reaches_clip.to_file(out_shp)
print("Wrote:", out_shp)

# # Convert geometry to WKT for CSV export
# reaches_clip_csv = reaches_clip.copy()
# reaches_clip_csv["geometry"] = reaches_clip_csv.geometry.to_wkt()

# reaches_clip_csv.to_csv(out_csv, index=False)
# print("Wrote:", out_csv)

out_geojson = out_dir / "dnipro_sword_reaches_clip.geojson"

# Web maps usually expect WGS84
reaches_clip_wgs84 = reaches_clip.to_crs(epsg=4326)

reaches_clip_wgs84.to_file(out_geojson, driver="GeoJSON")
print("Wrote:", out_geojson)

# If you prefer GeoPackage (often safer than Shapefile):
# reaches_clip.to_file(out_gpkg, layer="dnipro_reaches", driver="GPKG")
# print("Wrote:", out_gpkg)


Reaches CRS: EPSG:4326
Basin CRS: EPSG:4326
Input reaches: 6136
Clipped reaches: 842
Wrote: data\Dnipro_basin_shapefiles\derived\dnipro_sword_reaches_clip.shp
Wrote: data\Dnipro_basin_shapefiles\derived\dnipro_sword_reaches_clip.geojson


In [9]:
reaches_clip.head()

,x,y,reach_id,reach_len,n_nodes,wse,wse_var,width,width_var,facc,...,trib_flag,path_freq,path_order,path_segs,main_side,strm_order,end_reach,network,geometry,url
2343,33.542051,46.838603,22511300103,10517.863872,53,12.1,0.000000,4351.000000,164507.685706,488511.708554,...,0,39,262,200,0,5,0,2,"LINESTRING (33.47593 46.83353, 33.47632 46.833...",https://ihpwinsdata.blob.core.windows.net/data...
2267,32.361843,46.521577,22400900035,15336.123108,77,0.0,0.000867,426.500000,47086.428518,129.298721,...,0,-9999,-9999,944,1,-9999,3,2,"LINESTRING (32.37496 46.52541, 32.37526 46.525...",https://ihpwinsdata.blob.core.windows.net/data...
3111,32.369500,46.550351,22601000045,12699.288888,63,0.0,3.821328,278.163208,39592.084131,88.218163,...,0,41,262,201,0,5,3,2,"LINESTRING (32.38201 46.54688, 32.38202 46.546...",https://ihpwinsdata.blob.core.windows.net/data...
2274,32.530941,46.593655,22511100015,15609.133468,78,0.7,0.238765,225.000000,26398.122526,510466.468750,...,1,41,262,201,0,5,3,2,"LINESTRING (32.44571 46.55942, 32.44611 46.559...",https://ihpwinsdata.blob.core.windows.net/data...
2275,32.724987,46.633470,22511100055,18748.992721,94,1.4,0.001881,105.000000,5196.643614,219.899963,...,1,-9999,-9999,998,1,-9999,3,2,"LINESTRING (32.61399 46.6187, 32.61478 46.6176...",https://ihpwinsdata.blob.core.windows.net/data...
